# ML Zoomcamp 2026 — Homework 6

Data: `data/car_fuel_efficiency_2026.csv` (2026 pinned release).
Write-up: `ML_06_HW.md`.

Kernel: this folder `.venv`. Do not use `!pip` / `!wget`.


## 0) Imports


In [ ]:
# Point caches/temp to E: and cap BLAS threads so Jupyter stays on this drive and stays stable.

import os
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
# - Names: os.environ.setdefault
# - Role: set an env var only if missing (does not overwrite an existing value).
# - Purpose: keep matplotlib/temp/BLAS caches on E: and limit thread storms.
# - Now: first setup step before importing ML libraries.

# Import homework libraries: tables, trees/forests, RMSE, and XGBoost.

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
# - Names: DictVectorizer, root_mean_squared_error, train_test_split,
#   DecisionTreeRegressor, export_text, RandomForestRegressor, xgb
# - DictVectorizer: turns dict rows into a feature matrix (one-hot for categoricals).
# - root_mean_squared_error: validation score for regression.
# - train_test_split: random row split into two frames.
# - DecisionTreeRegressor / RandomForestRegressor: tree models for Q1–Q5.
# - export_text: prints the tree rules in plain text.
# - xgb: gradient boosting for Q6.
# - Now: tools ready; next cell loads and splits the CSV.


## 1) Load data and split

Fill missing values with zeros. Split with the exact calls from the homework.


In [ ]:
# Load CSV, fill NaNs with 0, split 60/20/20, then DictVectorize features.

df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
df = df.fillna(0)
# - Names: read_csv, fillna
# - read_csv: loads the pinned 2026 table into a DataFrame.
# - fillna(0): replaces missing numbers/categories with 0 (homework rule).
# - Now: raw table ready for the exact split recipe.

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
# - Names: train_test_split
# - Role: shuffle-split rows into two parts with a fixed seed.
# - Purpose: first 20% test, then 25% of the remaining 80% → val (overall 60/20/20).
# - Now: train/val rows fixed; test is unused for these homework questions.

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]
# - Names: .values, del column
# - .values: NumPy target arrays for sklearn/xgboost.
# - del: drop the label from features so it cannot leak into X.
# - Now: X tables hold only predictors.

train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
# - Names: to_dict, DictVectorizer, fit_transform, transform
# - to_dict(orient="records"): one dict per row for the vectorizer.
# - DictVectorizer(sparse=True): builds sparse one-hot / numeric columns.
# - fit_transform: learn feature names on train and build X_train.
# - transform: apply the same mapping to val (no new categories invented).
# - Now: matrices ready for tree / forest / boosting models.

df.shape, X_train.shape, X_val.shape


## Q1 — Decision tree, max_depth=1

Which original feature is used for splitting? Map a one-hot name such as `origin=USA` back to the original column.

- `vehicle_weight`
- `model_year`
- `origin`
- `fuel_type`

Answer:


In [ ]:
# Fit a stump (depth 1) and print which feature it splits on.

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=dv.feature_names_))
# - Names: DecisionTreeRegressor, fit, export_text, feature_names_
# - DecisionTreeRegressor(max_depth=1): one binary split only (a stump).
# - fit: learns that single threshold on training data.
# - export_text: human-readable rule tree; map one-hot names back to original columns.
# - Now: answer Q1 from the printed split feature.


## Q2 — Random forest RMSE

`n_estimators=10`, `random_state=1`. Validation RMSE.

- 0.1837
- 1.837
- 18.37
- 183.7

Answer:


In [ ]:
# Train a small random forest and report validation RMSE.

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)
# - Names: RandomForestRegressor, predict, root_mean_squared_error
# - RandomForestRegressor: averages many trees; n_estimators=10, seed=1, all CPU cores.
# - predict: validation predictions from the fitted forest.
# - root_mean_squared_error: sqrt of mean squared error vs y_val (pick closest form option).
# - Now: answer Q2 from this single RMSE number.


## Q3 — n_estimators

Try `[10, 50, 100, 150]`, `random_state=1`. Which value has the lowest validation RMSE (3 decimal places)?

- 10
- 50
- 100
- 150

Answer:


In [ ]:
# Sweep n_estimators; keep RMSE rounded to 3 decimals and pick the lowest.

scores = []
for n in [10, 50, 100, 150]:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, model.predict(X_val))
    scores.append((n, round(rmse, 3)))
scores
# - Names: RandomForestRegressor(n_estimators=...), round(rmse, 3)
# - n_estimators: how many trees are averaged in the forest.
# - round(..., 3): match the homework's "3 decimal places" comparison.
# - Now: choose the n with the smallest rounded validation RMSE (Q3).


## Q4 — max_depth by mean RMSE

`max_depth` in `[10, 15, 20, 25]`. For each, `n_estimators` in `[10, 50, 100, 150]`. `random_state=1`.

- 10
- 15
- 20
- 25

Answer:


In [ ]:
# For each max_depth, average RMSE across n_estimators; pick the best depth.

depth_scores = []
for depth in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1
        )
        model.fit(X_train, y_train)
        rmses.append(root_mean_squared_error(y_val, model.predict(X_val)))
    depth_scores.append((depth, round(float(np.mean(rmses)), 3)))
depth_scores
# - Names: max_depth, np.mean
# - max_depth: caps how deep each tree may grow (controls complexity).
# - np.mean: mean RMSE over the four n_estimators values at that depth.
# - Now: answer Q4 with the depth that has the lowest mean RMSE.


## Q5 — Most important feature

`n_estimators=10`, `max_depth=20`, `random_state=1`. Among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`.

Answer:


In [ ]:
# Fit the specified forest and rank feature_importances_ (Q5 choices only).

rf = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=dv.feature_names_)
importances.sort_values(ascending=False).head(10)
# - Names: feature_importances_, pd.Series, sort_values
# - feature_importances_: impurity-based importance per DictVectorizer column.
# - pd.Series(..., index=...): align scores with human-readable feature names.
# - sort_values(ascending=False): highest importance first; compare only the four options.
# - Now: pick the most important among the Q5 choices.


## Q6 — XGBoost eta

100 rounds. Compare `eta` 0.3 and 0.1 on validation RMSE.

- 0.3
- 0.1
- Both give equal value

Answer:


In [ ]:
# Build XGBoost DMatrix objects, then train 100 rounds for eta 0.3 and 0.1.

features = dv.feature_names_
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, "train"), (dval, "val")]
# - Names: xgb.DMatrix, watchlist
# - DMatrix: XGBoost's internal data format (features + labels + names).
# - watchlist: datasets logged during training (kept quiet via verbose_eval=False).
# - Now: shared matrices ready; only eta changes between runs.

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}
# - Names: eta, objective, seed
# - eta: learning rate (step size) for each boosting round.
# - objective "reg:squarederror": squared-error regression loss.
# - seed: reproducibility; other keys match the homework defaults.
# - Now: base params fixed before the eta sweep.

scores = {}
for eta in (0.3, 0.1):
    params = dict(xgb_params)
    params["eta"] = eta
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    scores[eta] = root_mean_squared_error(y_val, y_pred)
scores
# - Names: xgb.train, num_boost_round
# - xgb.train: runs boosting for num_boost_round=100 trees/rounds.
# - Purpose: compare validation RMSE for eta 0.3 vs 0.1 (Q6).
# - Now: lower RMSE wins; report that eta.
